# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/muhammadabdurrehmanmaqsood/flyrank-ml-internship-abdurrehman/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This notebook is the synthesis of `w01`–`w07`: the ranked, honest decline-detection model, the leakage confession test that caught a false 100% score, and the resulting action playbook. It mirrors the deployed paper at the live URL in `submission/paper_url.txt`.


## 1. Question

**Lane:** Refresh / Content Opportunity Scoring (see `w01_research_question.ipynb`).

**The decision this supports:** which specific, currently declining pages should the content team review first today?

**Who acts on it:** a content strategist opens the top-ranked page and decides whether to refresh, consolidate, or leave it, using the reason codes attached to each row.

**Cost of a wrong call:** ranking a low-value, seasonal, or permanently dead page at the top wastes reviewer hours that could go to a higher-value opportunity.

**What this work can claim:** a ranking of content by resemblance to past declining pages, usable as a prioritization aid.
**What it cannot claim:** that editing a flagged page will cause a traffic recovery — this is observational data, not a controlled experiment.


In [1]:
import pandas as pd

url = 'https://raw.githubusercontent.com/muhammadabdurrehmanmaqsood/flyrank-ml-internship-abdurrehman/main/data/raw/content_refresh_anonymized.csv'
df = pd.read_csv(url)

total_pages = len(df)
declining_pages = len(df[df['trend_direction'] == 'down'])
high_value_declining = len(df[(df['trend_direction'] == 'down') & (df['impressions_90d'] > 500)])

print(f"Total pages in starter dataset: {total_pages}")
print(f"Pages showing a downward trend: {declining_pages}")
print(f"Declining pages with >500 recent impressions (primary targets): {high_value_declining}")


Total pages in starter dataset: 30000
Pages showing a downward trend: 16262
Declining pages with >500 recent impressions (primary targets): 9956


## 2. Data

**Source:** FlyRank's production search-performance warehouse — 79 million rows of daily content performance across the full client base (`fact_content_daily_performance`). A single verified month (2026-03-01 to 2026-03-31, one row per client × content item × day) confirmed the warehouse's scale at 9.84M rows for that slice alone (see `w03_data_contract.ipynb`).

**Modeling slice:** a fixed, anonymized 30,000-row cross-section (`content_refresh_anonymized.csv`) — 32 clients, 3 content types (keyword article, feedly article, comparison article), each row a 90-day performance snapshot.

**Excluded, and why (public-safe):**
- `trend_direction`, `trend_pct` — the label is computed directly from these; including them leaks the answer.
- Client identities, URLs, raw query text — never present; all IDs are anonymized hashes.
- `impressions_last_30d`, `impressions_prev_30d` — retained in the raw file but excluded from the final model (see Methodology: label-sibling leakage).

**Data-quality fix applied:** `avg_position == 0` means "no ranking data," not rank zero. 1,205 rows were flagged and imputed from valid values only, rather than read as elite #1 rankings.


In [2]:
import numpy as np

df['has_avg_position'] = (df['avg_position'] != 0).astype(int)
n_no_position = (df['avg_position'] == 0).sum()
print(f"Rows where avg_position == 0 ('no data', fixed before modeling): {n_no_position} ({n_no_position/len(df):.1%})")
print(f"Clients: {df['client_id'].nunique()} | Content types: {sorted(df['content_type'].unique())}")


Rows where avg_position == 0 ('no data', fixed before modeling): 1205 (4.0%)
Clients: 32 | Content types: ['comparison article', 'feedly article', 'keyword article']


## 3. Methodology

**Label:** `is_declining_label = (trend_pct < 0)`.
**Baseline:** rule-based — stale (>180 days since update) AND slipping (position > 10) AND has valid position data.
**Validation design:** 80/20 split grouped by `client_id` (`GroupShuffleSplit`, seed 42) — no client appears in both train and test. A single holdout, not cross-validated (see Limitations).

**The leakage story:** an early version of this model, trained on every numeric column, scored precision/recall/accuracy of 1.000 on held-out data — a red flag on a real-world behavioral label, not a result to celebrate. Two features, `impressions_last_30d` and `impressions_prev_30d`, had coefficients that nearly canceled (-7.92 / +7.91) — the signature of features mechanically tied to how the label itself is computed.

**The confession test** below trains once with these suspect features, once without, everything else identical. A collapse from ~1.0 toward the base rate is the confession.


In [3]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score, recall_score, accuracy_score
from sklearn.impute import SimpleImputer

RANDOM_SEED = 42

df['has_word_count'] = df['word_count'].notna().astype(int)
df['word_count'] = df['word_count'].fillna(df['word_count'].median())
df['is_declining_label'] = (df['trend_pct'] < 0).astype(int)

def run_model(data, feature_cols, seed=RANDOM_SEED, split_col='client_id'):
    gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=seed)
    train_idx, test_idx = next(gss.split(data, groups=data[split_col]))
    train_df, test_df = data.iloc[train_idx].copy(), data.iloc[test_idx].copy()
    X_train, y_train = train_df[feature_cols], train_df['is_declining_label']
    X_test, y_test = test_df[feature_cols], test_df['is_declining_label']
    imputer = SimpleImputer(strategy='median', add_indicator=True)
    X_train_i = imputer.fit_transform(X_train)
    X_test_i = imputer.transform(X_test)
    model = LogisticRegression(max_iter=5000, random_state=seed)
    model.fit(X_train_i, y_train)
    preds = model.predict(X_test_i)
    return {
        'precision': precision_score(y_test, preds, zero_division=0),
        'recall': recall_score(y_test, preds),
        'accuracy': accuracy_score(y_test, preds),
        'base_rate': y_test.mean(),
    }

forbidden_always = ['is_declining_label', 'trend_direction', 'trend_pct', 'content_id', 'client_id', 'content_type']
suspects = ['impressions_last_30d', 'impressions_prev_30d']
all_numeric = [c for c in df.columns if c not in forbidden_always and pd.api.types.is_numeric_dtype(df[c])]

with_suspects = run_model(df, all_numeric)
without_suspects = run_model(df, [c for c in all_numeric if c not in suspects])

confession_table = pd.DataFrame({
    'Metric': ['Precision', 'Recall', 'Accuracy', 'Base Rate'],
    'WITH suspects (impressions_*_30d)': [round(with_suspects[k], 3) for k in ['precision','recall','accuracy','base_rate']],
    'WITHOUT suspects': [round(without_suspects[k], 3) for k in ['precision','recall','accuracy','base_rate']],
})
print("Leakage confession test — does the score collapse toward the base rate?\n")
print(confession_table.to_string(index=False))


Leakage confession test — does the score collapse toward the base rate?

   Metric  WITH suspects (impressions_*_30d)  WITHOUT suspects
Precision                              1.000             0.716
   Recall                              1.000             0.783
 Accuracy                              1.000             0.669
Base Rate                              0.628             0.628


## 4. Results (vs baseline)

Global precision/recall describe the whole test set; the queue is read top-down, so **precision@K** — how clean the first K rows are — is the metric that matters for a ranked list. The honest model below applies the `avg_position` fix from Section 2 on top of the leakage fix from Section 3.


In [4]:
df2 = df.copy()
df2.loc[df2['avg_position'] == 0, 'avg_position'] = np.nan
df2['avg_position'] = df2['avg_position'].fillna(df2['avg_position'].median())

features_clean = [c for c in all_numeric if c not in suspects] + ['has_avg_position']

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_SEED)
train_idx, test_idx = next(gss.split(df2, groups=df2['client_id']))
train_df, test_df = df2.iloc[train_idx].copy(), df2.iloc[test_idx].copy()
X_train, y_train = train_df[features_clean], train_df['is_declining_label']
X_test, y_test = test_df[features_clean], test_df['is_declining_label']

imputer = SimpleImputer(strategy='median', add_indicator=True)
X_train_i = imputer.fit_transform(X_train)
X_test_i = imputer.transform(X_test)
feat_names = imputer.get_feature_names_out(X_train.columns)

model = LogisticRegression(max_iter=20000, solver='liblinear', random_state=RANDOM_SEED)
model.fit(X_train_i, y_train)

probs = model.predict_proba(X_test_i)[:, 1]
preds = model.predict(X_test_i)
precision = precision_score(y_test, preds, zero_division=0)
recall = recall_score(y_test, preds)
accuracy = accuracy_score(y_test, preds)
base_rate = y_test.mean()

order = np.argsort(-probs)
y_sorted = y_test.values[order]
p_at_100 = y_sorted[:100].mean()
p_at_250 = y_sorted[:250].mean()

print("Honest model (final) — held-out test set, grouped split by client_id:")
print(f"  Global -- Precision: {precision:.3f} | Recall: {recall:.3f} | Accuracy: {accuracy:.3f} | Base rate: {base_rate:.3f}")
print(f"  Top-of-queue -- precision@100: {p_at_100:.3f} | precision@250: {p_at_250:.3f}")
print(f"\n  For reference -- rule-based baseline flagged 63 items directly (see w04_baseline_score.ipynb).")


Honest model (final) — held-out test set, grouped split by client_id:
  Global -- Precision: 0.713 | Recall: 0.801 | Accuracy: 0.673 | Base rate: 0.628
  Top-of-queue -- precision@100: 0.890 | precision@250: 0.868

  For reference -- rule-based baseline flagged 63 items directly (see w04_baseline_score.ipynb).


## 5. Limitations

- **Single holdout, not cross-validated** — precision@K would move somewhat under a different seed.
- **Coverage gap:** the client-grouped holdout used for precision@100/250 happened to land entirely on clients whose content is 100% "keyword article." The model trained on all 3 content types, but precision@K is directly demonstrated for one of them.
- **Not a calibrated probability** — `predicted_decline_probability` ranks well but is not a literal percentage chance.
- **No time-forward claim** — no calendar-date column exists to support a genuine past-vs-future split; this describes current resemblance to past declining content, not a forecast.
- **Global precision is modest** (71.5%, close to the 62.8% base rate) — trustworthy at the top of the queue, not row-by-row throughout.
- **No causal claim** — observational data; cannot say refreshing a page *causes* recovery.


In [5]:
# No computation needed — this section documents limitations already
# surfaced by the confession test (Section 3) and the coverage check
# below (which content types/clients the precision@K holdout actually covers).
covered_types = sorted(test_df['content_type'].unique())
covered_clients = test_df['client_id'].nunique()
print(f"Content types covered by this precision@K holdout: {covered_types}")
print(f"Clients covered: {covered_clients} of {df['client_id'].nunique()} total")


Content types covered by this precision@K holdout: ['keyword article']
Clients covered: 7 of 32 total


## 6. Ranked recommendations

The honest model is scored once on the held-out test set (never trained on) to build the queue. For each row, the top-2 features by signed contribution become plain-language reason codes.

**Priority order:**
1. Act on the top 100–250 of the queue first — precision@K (86–88%) is where the model is genuinely reliable.
2. Route rows with missing ranking data (`has_avg_position == 0`, ~1% of rows) to a data-verification step, never straight to an action.
3. Treat rows beyond the top few hundred as advisory only, given global precision near the base rate.
4. Re-validate before extending to clients or content types outside the training set.

**Never automate:** auto-publishing/editing content from this queue, auto-deprioritizing pages flagged only for missing data, or treating the ranked score as a guarantee.

**Monitoring / retrain triggers:** base-rate drift beyond ~5 points from 62.8%; precision@100 falling below ~0.75; a single feature's coefficient suddenly dominating again (the leakage signature reappearing); production batches containing content types or clients outside the known training pool.


In [6]:
# Build the ranked queue and reason codes on the held-out test set.
coefs = pd.Series(model.coef_[0], index=feat_names)
X_test_i_df = pd.DataFrame(X_test_i, columns=feat_names, index=test_df.index)
contributions = X_test_i_df * coefs
top_reason = contributions.abs().apply(lambda r: contributions.columns[np.argsort(-r.values)[:2]], axis=1)

queue = test_df.copy()
queue['predicted_decline_probability'] = probs
queue['reason_code_1'] = [r[0] for r in top_reason]
queue['reason_code_2'] = [r[1] for r in top_reason]
queue = queue.sort_values('predicted_decline_probability', ascending=False).reset_index(drop=True)
queue['rank'] = queue.index + 1

print("Top of the ranked queue (reason codes only -- no client-identifying columns shown):")
print(queue[['rank', 'predicted_decline_probability', 'reason_code_1', 'reason_code_2']].head(10).to_string(index=False))


Top of the ranked queue (reason codes only -- no client-identifying columns shown):
 rank  predicted_decline_probability reason_code_1         reason_code_2
    1                       1.000000    clicks_90d       clicks_last_30d
    2                       0.999999    clicks_90d       clicks_last_30d
    3                       0.999998    clicks_90d          sessions_90d
    4                       0.999982    clicks_90d          sessions_90d
    5                       0.999974    clicks_90d       clicks_last_30d
    6                       0.999965    clicks_90d       clicks_last_30d
    7                       0.999915    clicks_90d       clicks_last_30d
    8                       0.999586    clicks_90d       clicks_last_30d
    9                       0.999290    clicks_90d days_with_impressions
   10                       0.999186    clicks_90d       clicks_last_30d


## 7. Artifacts the paper embeds

Generates the three charts and the metrics summary the deployed page uses: the leakage-collapse story, the precision@K climb, and the honest model's coefficient spread.


In [7]:
import os
import json
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

os.makedirs('../figures', exist_ok=True)
os.makedirs('../../docs/img', exist_ok=True)
os.makedirs('../outputs', exist_ok=True)

# Chart 1 -- leakage story
fig, ax = plt.subplots(figsize=(7.5, 4.6))
labels = ['Base rate\n(majority guess)', 'Leaky model\n(pre-audit)', 'Honest model\n(global)', 'Honest model\n(precision@100)']
values = [base_rate, 1.000, accuracy, p_at_100]
colors = ['#9AA5B1', '#D64545', '#4C72B0', '#2E8B57']
bars = ax.bar(labels, values, color=colors, width=0.6)
for b, v in zip(bars, values):
    ax.text(b.get_x() + b.get_width()/2, v + 0.015, f"{v:.1%}", ha='center', fontsize=10, fontweight='bold')
ax.set_ylim(0, 1.12); ax.set_ylabel('Score')
ax.set_title('From a suspicious 100% to an honest, useful signal')
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig('../figures/chart1_leakage_story.png', dpi=150)
plt.savefig('../../docs/img/chart1_leakage_story.png', dpi=150)
plt.close()

# Chart 2 -- precision@K
fig, ax = plt.subplots(figsize=(7.5, 4.6))
ks = ['Base rate', 'Global\nprecision', 'precision@250', 'precision@100']
vals = [base_rate, precision, p_at_250, p_at_100]
colors2 = ['#9AA5B1', '#4C72B0', '#DD8452', '#2E8B57']
bars = ax.bar(ks, vals, color=colors2, width=0.55)
for b, v in zip(bars, vals):
    ax.text(b.get_x() + b.get_width()/2, v + 0.015, f"{v:.1%}", ha='center', fontsize=10, fontweight='bold')
ax.set_ylim(0, 1.0); ax.set_ylabel('Precision')
ax.set_title('The top of the queue is far cleaner than the global score')
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig('../figures/chart2_precision_at_k.png', dpi=150)
plt.savefig('../../docs/img/chart2_precision_at_k.png', dpi=150)
plt.close()

# Chart 3 -- coefficient spread
top_coefs = pd.Series(model.coef_[0], index=feat_names).abs().sort_values(ascending=False).head(8)
fig, ax = plt.subplots(figsize=(7.5, 4.6))
ax.barh(top_coefs.index[::-1], top_coefs.values[::-1], color='#4C72B0')
ax.set_xlabel('|Coefficient|')
ax.set_title('Honest model: no single feature dominates')
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig('../figures/chart3_coefficients.png', dpi=150)
plt.savefig('../../docs/img/chart3_coefficients.png', dpi=150)
plt.close()

summary = {
    'total_rows': int(len(df)), 'clients': int(df['client_id'].nunique()),
    'declining_pages': int((df['trend_direction'] == 'down').sum()),
    'high_value_declining': int(((df['trend_direction'] == 'down') & (df['impressions_90d'] > 500)).sum()),
    'honest_precision': round(precision, 3), 'honest_recall': round(recall, 3),
    'honest_accuracy': round(accuracy, 3), 'base_rate': round(base_rate, 3),
    'precision_at_100': round(p_at_100, 3), 'precision_at_250': round(p_at_250, 3),
}
with open('../outputs/paper_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)
print("Artifacts written: 3 charts (work/figures/ + docs/img/) and work/outputs/paper_summary.json")
print(json.dumps(summary, indent=2))


Artifacts written: 3 charts (work/figures/ + docs/img/) and work/outputs/paper_summary.json
{
  "total_rows": 30000,
  "clients": 32,
  "declining_pages": 16262,
  "high_value_declining": 9956,
  "honest_precision": 0.713,
  "honest_recall": 0.801,
  "honest_accuracy": 0.673,
  "base_rate": 0.628,
  "precision_at_100": 0.89,
  "precision_at_250": 0.868
}


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
